[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

# ONNX Proto Structure — Deep Dive

| # | Section | Description |
|---|---------|-------------|
| 1 | [The ONNX Message Hierarchy](#1-the-onnx-message-hierarchy) | Top-level architecture overview |
| 2 | [ModelProto — The Root Container](#2-modelproto--the-root-container) | Fields, IR version, opset imports |
| 3 | [GraphProto — The Computation Graph](#3-graphproto--the-computation-graph) | Nodes, inputs, outputs, initializers |
| 4 | [NodeProto — Individual Operations](#4-nodeproto--individual-operations) | op_type, attributes, domains |
| 5 | [TensorProto — Weight Storage](#5-tensorproto--weight-storage) | Data types, raw_data, external data |
| 6 | [ValueInfoProto and TypeProto](#6-valueinfoproto-and-typeproto) | Shape info, type system |
| 7 | [AttributeProto — Static Parameters](#7-attributeproto--static-parameters) | Attribute types, nested graphs |
| 8 | [Message Size Computation](#8-message-size-computation) | Formal size analysis |
| 9 | [Visualizing the Full Hierarchy](#9-visualizing-the-full-hierarchy) | Interactive tree visualization |
| 10 | [Building a Complete Model Programmatically](#10-building-a-complete-model) | End-to-end construction |
| 11 | [Formal Specification and Invariants](#11-formal-specification-and-invariants) | Notation and invariants |
| 12 | [Key Takeaways](#12-key-takeaways) | Core concepts |

In [ ]:
# !pip install onnx numpy matplotlib networkx --quiet

import onnx
from onnx import helper, TensorProto, checker, numpy_helper
from onnx import AttributeProto, GraphProto, NodeProto, ModelProto
import numpy as np
import matplotlib.pyplot as plt

## 1. The ONNX Message Hierarchy

An ONNX model file (`.onnx`) is a single serialized `ModelProto` message. This top-level container holds the computation graph, metadata, and versioning information in a nested hierarchy of Protobuf messages.

### Formal Containment Relation

The hierarchy can be described formally as a tree of typed containers:

$$\texttt{ModelProto} \supset \texttt{GraphProto} \supset \{\texttt{NodeProto}^*, \texttt{TensorProto}^*, \texttt{ValueInfoProto}^*\}$$

where $^*$ denotes `repeated` (zero or more instances).

![ONNX Protobuf Structure](assets/onnx_protobuf_structure.png)

### ASCII Architecture Diagram

```
┌──────────────────────────────────────────────────────────────────┐
│  ModelProto                                                      │
│  ├── ir_version: int64                                           │
│  ├── opset_import: OperatorSetIdProto[]                          │
│  │   ├── domain: string   ("" = default ai.onnx)                │
│  │   └── version: int64   (e.g. 17)                              │
│  ├── producer_name / producer_version: string                    │
│  ├── domain: string       (model namespace)                      │
│  ├── model_version: int64 (user-defined)                         │
│  ├── doc_string: string                                          │
│  ├── metadata_props: StringStringEntryProto[]                    │
│  ├── functions: FunctionProto[]    ◄── model-local op defs       │
│  │                                                               │
│  └── graph: GraphProto                                           │
│      ├── name: string                                            │
│      ├── input: ValueInfoProto[]    ◄── runtime inputs + weights │
│      ├── output: ValueInfoProto[]   ◄── graph result tensors     │
│      ├── initializer: TensorProto[] ◄── trained weight values    │
│      ├── value_info: ValueInfoProto[]◄── intermediate shapes     │
│      │                                                           │
│      └── node: NodeProto[]          ◄── topologically sorted     │
│          ├── op_type: string                                     │
│          ├── domain: string                                      │
│          ├── input / output: string[]                            │
│          ├── name: string                                        │
│          └── attribute: AttributeProto[]                         │
│              ├── scalars: f, i, s, t, g                          │
│              ├── lists: floats, ints, strings, tensors, graphs   │
│              └── (g/graphs may contain nested GraphProto!)       │
└──────────────────────────────────────────────────────────────────┘
```

Each level serves a distinct purpose: `ModelProto` provides context and versioning; `GraphProto` defines the data-flow graph; `NodeProto` specifies individual operations; and `TensorProto`/`ValueInfoProto` describe the data that flows between nodes.

Understanding this hierarchy is essential for programmatically constructing, inspecting, modifying, and debugging ONNX models. Every interaction with the `onnx` Python API maps directly to navigating this tree.

## 2. ModelProto — The Root Container

`ModelProto` is the outermost message, containing everything needed to describe a deployable model. Its key fields with their proto field numbers:

| Field | Tag | Type | Purpose |
|-------|-----|------|---------|
| `ir_version` | 1 | `int64` | ONNX IR spec version (currently 8–10) |
| `opset_import` | 8 | `OperatorSetIdProto[]` | Which operator sets the model uses |
| `producer_name` | 2 | `string` | Tool that created the model |
| `producer_version` | 3 | `string` | Version of the producing tool |
| `domain` | 4 | `string` | Reverse-DNS model domain |
| `model_version` | 5 | `int64` | User-defined version number |
| `doc_string` | 6 | `string` | Human-readable documentation |
| `graph` | 7 | `GraphProto` | **The actual computation graph** |
| `metadata_props` | 14 | `StringStringEntryProto[]` | Key-value metadata pairs |
| `training_info` | 20 | `TrainingInfoProto[]` | Training-specific info |
| `functions` | 25 | `FunctionProto[]` | Model-local function definitions |

### OpSet Import Semantics

The **opset import** is critically important. A model declares which versions of which operator domains it uses:

$$\text{OpSetImport} = \{(d_1, v_1), (d_2, v_2), \ldots, (d_k, v_k)\}$$

where $d_i$ is a domain string (empty string `""` for the default ONNX domain `ai.onnx`) and $v_i$ is the opset version. A runtime must support **all** declared opsets to execute the model.

### The OpSet Resolution Rule

For any `NodeProto` with `domain = d` and `op_type = t`, the operator schema used is:

$$\text{Schema}(t, d) = \text{get\_schema}(t,\; v_d,\; d)$$

where $v_d$ is the version imported for domain $d$. If no import exists for domain $d$, the model is invalid.

```
ModelProto.opset_import resolution:

  opset_import: [ ("", 17), ("ai.onnx.ml", 3) ]
                    │                │
                    ▼                ▼
  Node(domain="")    → uses ai.onnx opset 17 schemas
  Node(domain="ai.onnx.ml") → uses ml domain opset 3 schemas
  Node(domain="custom")     → ERROR: no import for "custom"
```

In [ ]:
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, 3, 224, 224])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [1, 1000])

W = numpy_helper.from_array(np.random.randn(1000, 3*224*224).astype(np.float32)[:, :10], name="W")

node = helper.make_node("MatMul", ["X_flat", "W"], ["Y"])
reshape = helper.make_node("Reshape", ["X", "shape"], ["X_flat"])
shape_init = numpy_helper.from_array(np.array([1, -1], dtype=np.int64), name="shape")

graph = helper.make_graph([reshape, node], "classifier",
                          [X], [Y], initializer=[W, shape_init])
model = helper.make_model(graph, opset_imports=[
    helper.make_opsetid("", 17),
    helper.make_opsetid("ai.onnx.ml", 3),
])
model.ir_version = 9
model.producer_name = "onnx_tutorial"
model.producer_version = "1.0.0"
model.domain = "com.example.tutorial"
model.model_version = 1
model.doc_string = "Demo classifier for Proto Structure deep dive"

entry = model.metadata_props.add()
entry.key = "author"
entry.value = "ONNX Tutorial"
entry2 = model.metadata_props.add()
entry2.key = "license"
entry2.value = "MIT"

print("=== ModelProto Fields ===")
print(f"  ir_version:       {model.ir_version}")
print(f"  producer_name:    {model.producer_name}")
print(f"  producer_version: {model.producer_version}")
print(f"  domain:           {model.domain}")
print(f"  model_version:    {model.model_version}")
print(f"  doc_string:       {model.doc_string}")
print(f"  opset_import:")
for oi in model.opset_import:
    print(f"    domain={oi.domain!r:>15}, version={oi.version}")
print(f"  metadata_props:")
for p in model.metadata_props:
    print(f"    {p.key} = {p.value}")
print(f"  graph.name:       {model.graph.name}")
print(f"  graph.node count: {len(model.graph.node)}")

## 3. GraphProto — The Computation Graph

`GraphProto` is the heart of an ONNX model. It defines a **directed acyclic graph (DAG)** where nodes are operations and edges are named tensors.

### Formal Graph Definition

Let $G = (V, E)$ be the computation graph where:
- $V = \{n_1, n_2, \ldots, n_k\}$ is the ordered set of `NodeProto` entries
- $E \subseteq V \times V$ is defined implicitly: $(n_i, n_j) \in E$ iff some output name of $n_i$ appears in the input list of $n_j$

The graph must be a **DAG** — no cycles are permitted. Nodes are listed in **topological order**: for every edge $(n_i, n_j)$, node $n_i$ appears before $n_j$ in the node list.

### Graph Components

```
GraphProto
│
├── input: ValueInfoProto[]     ─── All named tensors entering the graph
│   │                               (both runtime feeds AND weight names)
│   │
├── initializer: TensorProto[]  ─── Pre-filled tensor values (weights)
│   │                               Names overlap with input[]
│   │
├── node: NodeProto[]           ─── Operations in topological order
│   │                               Each consumes inputs, produces outputs
│   │
├── output: ValueInfoProto[]    ─── Final result tensor(s)
│   │
└── value_info: ValueInfoProto[]─── Shape/type info for intermediate tensors
                                    (populated by shape inference)
```

### Input/Initializer Overlap Convention

An important subtlety: tensor names that appear in **both** `graph.input` and `graph.initializer` represent **optional overridable weights**. The initializer provides a default value, but the runtime may accept a user-supplied value instead. Tensor names that appear **only** in `graph.input` are **required runtime feeds**.

$$\text{required\_feeds} = \{t \mid t \in \text{input\_names} \wedge t \notin \text{initializer\_names}\}$$

$$\text{overridable\_weights} = \text{input\_names} \cap \text{initializer\_names}$$

### Edge Semantics

Edges are **named tensors** — strings that connect node outputs to node inputs. The naming must satisfy:

1. **Unique production**: Each tensor name is produced by exactly one source (either a graph input or a node output)
2. **Valid consumption**: Every tensor name consumed by a node input must be produced by a prior source (topological constraint)
3. **Type compatibility**: The producer's output type must match the consumer's expected input type

In [ ]:
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 784])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", 10])

W1 = numpy_helper.from_array(np.random.randn(784, 128).astype(np.float32), name="W1")
b1 = numpy_helper.from_array(np.zeros(128, dtype=np.float32), name="b1")
W2 = numpy_helper.from_array(np.random.randn(128, 10).astype(np.float32), name="W2")
b2 = numpy_helper.from_array(np.zeros(10, dtype=np.float32), name="b2")

nodes = [
    helper.make_node("MatMul", ["X", "W1"], ["mm1"]),
    helper.make_node("Add", ["mm1", "b1"], ["z1"]),
    helper.make_node("Relu", ["z1"], ["a1"]),
    helper.make_node("MatMul", ["a1", "W2"], ["mm2"]),
    helper.make_node("Add", ["mm2", "b2"], ["z2"]),
    helper.make_node("Softmax", ["z2"], ["Y"], axis=1),
]

graph = helper.make_graph(nodes, "two_layer_mlp", [X], [Y],
                          initializer=[W1, b1, W2, b2])
mlp_model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
checker.check_model(mlp_model)

g = mlp_model.graph
print("=== GraphProto Fields ===")
print(f"  name:           {g.name}")
print(f"  nodes:          {len(g.node)} operations")
print(f"  inputs:         {[i.name for i in g.input]}")
print(f"  outputs:        {[o.name for o in g.output]}")
print(f"  initializers:   {[init.name for init in g.initializer]}")

init_names = {init.name for init in g.initializer}
input_names = {i.name for i in g.input}
required_feeds = input_names - init_names
print(f"\n  Required runtime feeds:   {required_feeds}")
print(f"  Overridable weights:      {input_names & init_names}")

print(f"\n  Node details (topological order):")
for i, n in enumerate(g.node):
    print(f"    [{i}] {n.op_type:>10}: {list(n.input)} -> {list(n.output)}")

## 4. NodeProto — Individual Operations

Each `NodeProto` represents a single operation in the computation graph. A node's semantics are uniquely determined by its `(domain, op_type, opset_version)` triple.

### NodeProto Fields

| Field | Type | Purpose |
|-------|------|---------|
| `op_type` | `string` | Operator name (e.g., `Conv`, `MatMul`, `Relu`) |
| `input` | `string[]` | Ordered list of input tensor names |
| `output` | `string[]` | Ordered list of output tensor names |
| `name` | `string` | Optional human-readable node identifier |
| `domain` | `string` | Operator domain (empty for default ONNX) |
| `attribute` | `AttributeProto[]` | Static parameters (kernel_shape, pads, etc.) |
| `doc_string` | `string` | Optional documentation |

### Positional Input/Output Convention

**Input/output ordering is positional**, defined by the operator schema:

```
Conv operator schema:
  input[0] = X   (feature map)      [required]
  input[1] = W   (kernel weights)   [required]
  input[2] = B   (bias)             [optional]

  output[0] = Y  (convolution result)
```

**Optional inputs** are represented by empty strings: if a node doesn't use an optional input, its entry in the `input` list is `""`.

### Node Identity

A node's behavior is fully determined by:

$$\text{Behavior}(n) = f_{\text{op\_type}}^{(\text{domain}, v_{\text{domain}})}\bigl(\text{inputs}(n),\; \text{attrs}(n)\bigr)$$

where $f$ is the operator function selected by the imported opset version $v_{\text{domain}}$ for the node's domain.

In [ ]:
nodes_demo = [
    helper.make_node("Conv", ["X", "W", "B"], ["Y"],
                     kernel_shape=[3, 3], pads=[1, 1, 1, 1],
                     strides=[1, 1], dilations=[1, 1], group=1),
    helper.make_node("BatchNormalization", ["Y", "scale", "bias", "mean", "var"],
                     ["bn_out"], epsilon=1e-5, momentum=0.9),
    helper.make_node("Relu", ["bn_out"], ["relu_out"]),
    helper.make_node("MaxPool", ["relu_out"], ["pool_out"],
                     kernel_shape=[2, 2], strides=[2, 2]),
    helper.make_node("Gemm", ["flat", "Wfc", "bfc"], ["logits"],
                     alpha=1.0, beta=1.0, transA=0, transB=0),
]

for node in nodes_demo:
    print(f"\n{'='*55}")
    print(f"  op_type:    {node.op_type}")
    print(f"  domain:     {node.domain!r}")
    print(f"  inputs:     {list(node.input)}")
    print(f"  outputs:    {list(node.output)}")
    if node.attribute:
        print(f"  attributes:")
        for attr in node.attribute:
            if attr.type == AttributeProto.INT:
                print(f"    {attr.name} = {attr.i}")
            elif attr.type == AttributeProto.INTS:
                print(f"    {attr.name} = {list(attr.ints)}")
            elif attr.type == AttributeProto.FLOAT:
                print(f"    {attr.name} = {attr.f}")
            elif attr.type == AttributeProto.FLOATS:
                print(f"    {attr.name} = {list(attr.floats)}")
            elif attr.type == AttributeProto.STRING:
                print(f"    {attr.name} = {attr.s.decode()}")
            else:
                print(f"    {attr.name} (type={attr.type})")

## 5. TensorProto — Weight Storage

`TensorProto` stores constant tensors — model weights, biases, and other precomputed data. The tensor's data can be stored in several ways:

### Storage Methods

| Storage field | When used | Efficiency |
|--------------|-----------|------------|
| `raw_data` | Binary blob (recommended) | Most compact: exactly $s \cdot \prod d_i$ bytes |
| `float_data` | Float32 values as repeated field | Readable but larger due to varint overhead |
| `int32_data` | Int32 values | For integer tensors |
| `int64_data` | Int64 values | For shape tensors |
| `external_data` | Separate file reference | For large models (>2GB) |

### ONNX Data Type Enumeration

| Value | Name | Size (bytes) | NumPy dtype | Description |
|-------|------|-------------|-------------|-------------|
| 1 | `FLOAT` | 4 | `float32` | IEEE 754 single precision |
| 2 | `UINT8` | 1 | `uint8` | Unsigned 8-bit integer |
| 3 | `INT8` | 1 | `int8` | Signed 8-bit integer |
| 5 | `INT16` | 2 | `int16` | Signed 16-bit integer |
| 6 | `INT32` | 4 | `int32` | Signed 32-bit integer |
| 7 | `INT64` | 8 | `int64` | Signed 64-bit integer |
| 9 | `BOOL` | 1 | `bool` | Boolean |
| 10 | `FLOAT16` | 2 | `float16` | IEEE 754 half precision |
| 11 | `DOUBLE` | 8 | `float64` | IEEE 754 double precision |
| 16 | `BFLOAT16` | 2 | N/A | Brain floating-point |

### Raw Data Size Formula

The total raw size of a tensor with shape $(d_1, d_2, \ldots, d_n)$ and element size $s$ bytes is:

$$\text{raw\_bytes} = s \cdot \prod_{i=1}^{n} d_i$$

For a typical ResNet-50 with ~25.6M float32 parameters:

$$\text{raw\_bytes} = 4 \times 25{,}600{,}000 = 102{,}400{,}000 \;\text{bytes} \approx 97.7 \;\text{MB}$$

### External Data Format

When tensors exceed the Protobuf size limit, they are stored externally:

```
TensorProto (in .onnx file)           External file
┌─────────────────────────┐           ┌──────────────────┐
│ name: "conv1.weight"    │           │                  │
│ data_location: EXTERNAL │──────────▶│  raw bytes at    │
│ external_data:          │           │  offset with     │
│   location: weights.bin │           │  specified length│
│   offset: 0             │           │                  │
│   length: 36864         │           └──────────────────┘
└─────────────────────────┘
```

In [ ]:
dtype_names = {1: 'FLOAT', 2: 'UINT8', 3: 'INT8', 5: 'INT16',
               6: 'INT32', 7: 'INT64', 10: 'FLOAT16', 11: 'DOUBLE'}

tensors = {
    'float32': numpy_helper.from_array(np.random.randn(3, 4).astype(np.float32), name="f32"),
    'float64': numpy_helper.from_array(np.random.randn(3, 4).astype(np.float64), name="f64"),
    'int32':   numpy_helper.from_array(np.array([[1, 2], [3, 4]], dtype=np.int32), name="i32"),
    'int64':   numpy_helper.from_array(np.array([10, 20, 30], dtype=np.int64), name="i64"),
    'uint8':   numpy_helper.from_array(np.array([0, 128, 255], dtype=np.uint8), name="u8"),
}

print(f"{'Name':<10} | {'DataType':<10} | {'Dims':<15} | {'raw_data (B)':>12} | {'Computed':>10}")
print("-" * 68)
for name, t in tensors.items():
    dims = list(t.dims)
    raw_len = len(t.raw_data) if t.raw_data else 0
    arr = numpy_helper.to_array(t)
    expected = arr.nbytes
    dt_name = dtype_names.get(t.data_type, str(t.data_type))
    print(f"{name:<10} | {dt_name:<10} | {str(dims):<15} | {raw_len:>12} | {expected:>10}")

print("\nraw_data stores the tensor as a flat byte array in little-endian order.")

## 6. ValueInfoProto and TypeProto

`ValueInfoProto` associates a tensor name with its type information. It is the primary mechanism for communicating shape and data type expectations between nodes.

### Structure

```
ValueInfoProto
├── name: string              ─── matches edge names in the graph
├── doc_string: string        ─── optional documentation
└── type: TypeProto
    └── tensor_type: TypeProto.Tensor
        ├── elem_type: int32  ─── TensorProto.DataType enum value
        └── shape: TensorShapeProto
            └── dim: repeated Dimension
                ├── dim_value: int64   ─── fixed dimension (e.g., 224)
                └── dim_param: string  ─── symbolic dimension (e.g., "batch")
```

### TypeProto Variants

`TypeProto` uses a `oneof` to represent different value types:
- **Tensor type**: element data type + shape (the most common)
- **Sequence type**: ordered collection of tensors
- **Map type**: key-value mapping
- **Optional type**: nullable wrapper

### Shape Representation — Fixed vs Symbolic Dimensions

A tensor shape is a list of `Dimension` entries, each being either:
- A **fixed integer** (`dim_value`): e.g., $224$ for a known spatial dimension
- A **symbolic string** (`dim_param`): e.g., `"batch"` for a dynamic dimension

$$\text{Shape} = [d_1, d_2, \ldots, d_n] \quad \text{where } d_i \in \mathbb{Z}^+ \cup \Sigma^*$$

where $\Sigma^*$ is the set of all symbolic name strings. The shape algebra rule:

$$\text{dim\_match}(a, b) = \begin{cases} \text{true} & \text{if } a = b \text{ (both fixed, same value)} \\ \text{true} & \text{if } a \in \Sigma^* \text{ or } b \in \Sigma^* \\ \text{false} & \text{otherwise} \end{cases}$$

In [ ]:
value_infos = [
    helper.make_tensor_value_info("image", TensorProto.FLOAT, [1, 3, 224, 224]),
    helper.make_tensor_value_info("tokens", TensorProto.INT64, ["batch", "seq_len"]),
    helper.make_tensor_value_info("logits", TensorProto.FLOAT, ["batch", 1000]),
    helper.make_tensor_value_info("hidden", TensorProto.FLOAT, ["batch", "seq_len", 768]),
    helper.make_tensor_value_info("scalar", TensorProto.FLOAT, []),
]

print(f"{'Name':<10} | {'ElemType':<10} | {'Rank':>4} | Shape")
print("-" * 60)
for vi in value_infos:
    tensor_type = vi.type.tensor_type
    elem_type = dtype_names.get(tensor_type.elem_type, str(tensor_type.elem_type))
    dims = []
    if tensor_type.HasField('shape'):
        for d in tensor_type.shape.dim:
            if d.dim_param:
                dims.append(d.dim_param)
            else:
                dims.append(d.dim_value)
    rank = len(dims)
    print(f"{vi.name:<10} | {elem_type:<10} | {rank:>4} | {dims}")

## 7. AttributeProto — Static Parameters

`AttributeProto` carries static configuration for operators — values that are fixed at model creation time, not computed from input tensors. ONNX supports a rich set of attribute types:

### Attribute Type System

| Type enum | Value | Python field | Example use |
|-----------|-------|-------------|-------------|
| `FLOAT` | 1 | `.f` | `epsilon=1e-5` in BatchNorm |
| `INT` | 2 | `.i` | `axis=1` in Softmax |
| `STRING` | 3 | `.s` (bytes) | `mode=b"nearest"` in Resize |
| `TENSOR` | 4 | `.t` (TensorProto) | Constant values |
| `GRAPH` | 5 | `.g` (GraphProto) | Subgraphs in If/Loop |
| `FLOATS` | 6 | `.floats` | Multiple float values |
| `INTS` | 7 | `.ints` | `kernel_shape=[3,3]` |
| `STRINGS` | 8 | `.strings` | String lists |
| `TENSORS` | 9 | `.tensors` | Multiple tensors |
| `GRAPHS` | 10 | `.graphs` | Multiple subgraphs |

### Nested Graphs — Control Flow

The `GRAPH` type is particularly powerful — it enables control flow operators like `If` (conditional) and `Loop` (iteration) to contain nested computation graphs:

```
If operator:
┌──────────────────────────────────┐
│ NodeProto (op_type="If")         │
│   input: ["condition"]           │
│   attribute:                     │
│     then_branch: GraphProto ─────┼──▶ { nodes for true case }
│     else_branch: GraphProto ─────┼──▶ { nodes for false case }
│   output: ["result"]             │
└──────────────────────────────────┘
```

This is how ONNX represents dynamic control flow without breaking the DAG constraint at the top level — subgraphs are "inlined" as attribute values.

In [ ]:
attr_type_names = {
    1: 'FLOAT', 2: 'INT', 3: 'STRING', 4: 'TENSOR',
    5: 'GRAPH', 6: 'FLOATS', 7: 'INTS', 8: 'STRINGS',
}

conv_node = helper.make_node(
    "Conv", ["X", "W"], ["Y"],
    kernel_shape=[3, 3],
    pads=[1, 1, 1, 1],
    strides=[2, 2],
    dilations=[1, 1],
    group=1,
)

print("Conv node attributes:")
print(f"{'Name':<15} | {'Type':<8} | Value")
print("-" * 50)
for attr in conv_node.attribute:
    type_name = attr_type_names.get(attr.type, f'type={attr.type}')
    if attr.type == AttributeProto.INT:
        val = attr.i
    elif attr.type == AttributeProto.INTS:
        val = list(attr.ints)
    elif attr.type == AttributeProto.FLOAT:
        val = attr.f
    elif attr.type == AttributeProto.STRING:
        val = attr.s.decode()
    else:
        val = '...'
    print(f"{attr.name:<15} | {type_name:<8} | {val}")

bn_node = helper.make_node(
    "BatchNormalization", ["X", "s", "b", "m", "v"], ["Y"],
    epsilon=1e-5, momentum=0.9,
)
print("\nBatchNormalization attributes:")
for attr in bn_node.attribute:
    type_name = attr_type_names.get(attr.type, f'type={attr.type}')
    print(f"  {attr.name}: {type_name} = {attr.f}")

## 8. Message Size Computation

Understanding how Protobuf message sizes compose is essential for predicting file sizes and optimizing model storage.

### Per-Field Size Formula

Each field contributes to the total message size:

$$S_{\text{field}} = |\text{tag}| + |\text{len\_prefix}| + |\text{data}|$$

where:
- $|\text{tag}|$ = varint-encoded tag size (1–2 bytes for field numbers 1–2047)
- $|\text{len\_prefix}|$ = varint-encoded length (0 for varints/fixed, 1–5 for length-delimited)
- $|\text{data}|$ = actual data size

### Total Message Size

$$S_{\text{message}} = \sum_{i=1}^{N} (|\text{tag}_i| + |\text{len}_i| + |\text{data}_i|)$$

### ONNX Model Size Breakdown

For a typical neural network model:

$$S_{\text{model}} = S_{\text{metadata}} + S_{\text{graph\_structure}} + S_{\text{weights}}$$

where:

$$S_{\text{metadata}} = O(1) \quad \text{(producer, version, doc strings — typically < 1KB)}$$

$$S_{\text{graph\_structure}} = O(N_{\text{nodes}} + N_{\text{edges}}) \quad \text{(node definitions, shape info)}$$

$$S_{\text{weights}} = \sum_{j=1}^{N_{\text{init}}} s_j \cdot \prod_{k} d_j^{(k)} \quad \text{(raw tensor data)}$$

For large models, $S_{\text{weights}}$ completely dominates:

$$\frac{S_{\text{weights}}}{S_{\text{model}}} \xrightarrow{\text{large models}} 1$$

In [ ]:
def analyze_model_size(model: onnx.ModelProto) -> dict:
    """Break down model serialized size by component."""
    total = len(model.SerializeToString())
    weight_bytes = sum(len(init.raw_data) if init.raw_data else
                       numpy_helper.to_array(init).nbytes
                       for init in model.graph.initializer)
    graph_no_init = onnx.ModelProto()
    graph_no_init.CopyFrom(model)
    del graph_no_init.graph.initializer[:]
    structure = len(graph_no_init.SerializeToString())
    overhead = total - weight_bytes
    return {
        'total': total,
        'weights': weight_bytes,
        'structure': structure,
        'overhead': overhead,
        'weight_pct': weight_bytes / total * 100 if total > 0 else 0,
    }

analysis = analyze_model_size(mlp_model)
print("=== MLP Model Size Breakdown ===")
print(f"  Total serialized:  {analysis['total']:>10,} bytes")
print(f"  Weight data:       {analysis['weights']:>10,} bytes ({analysis['weight_pct']:.1f}%)")
print(f"  Graph structure:   {analysis['structure']:>10,} bytes")
print(f"  Proto overhead:    {analysis['overhead']:>10,} bytes")

print("\n=== Per-initializer breakdown ===")
for init in mlp_model.graph.initializer:
    dims = list(init.dims)
    nbytes = len(init.raw_data) if init.raw_data else numpy_helper.to_array(init).nbytes
    print(f"  {init.name:<10s}: shape={str(dims):<15s} -> {nbytes:>10,} bytes")

## 9. Visualizing the Full Hierarchy

Let's create comprehensive visualizations of an ONNX model's proto structure.

![ONNX Protobuf Structure](assets/onnx_protobuf_structure.png)

![Linear Regression Graph](assets/dot_linreg.png)

In [ ]:
import networkx as nx

def visualize_model_graph(model: onnx.ModelProto, title: str = "ONNX Computation Graph"):
    G = nx.DiGraph()
    g = model.graph
    init_names = {init.name for init in g.initializer}

    for inp in g.input:
        if inp.name in init_names:
            G.add_node(inp.name, type='weight', label=f"W: {inp.name}")
        else:
            G.add_node(inp.name, type='input', label=f"IN: {inp.name}")

    for i, node in enumerate(g.node):
        node_id = f"op_{i}_{node.op_type}"
        G.add_node(node_id, type='op', label=node.op_type)
        for inp in node.input:
            if inp:
                if inp not in G:
                    G.add_node(inp, type='intermediate', label=inp)
                G.add_edge(inp, node_id)
        for out in node.output:
            G.add_node(out, type='intermediate', label=out)
            G.add_edge(node_id, out)

    for out in g.output:
        if out.name in G:
            G.nodes[out.name]['type'] = 'output'
            G.nodes[out.name]['label'] = f"OUT: {out.name}"

    color_map = {'input': '#4CAF50', 'weight': '#FF9800', 'op': '#2196F3',
                 'intermediate': '#9E9E9E', 'output': '#F44336'}
    colors = [color_map.get(G.nodes[n].get('type', 'intermediate'), '#9E9E9E') for n in G]
    labels = {n: G.nodes[n].get('label', n) for n in G}

    fig, ax = plt.subplots(figsize=(14, 8))
    pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
    nx.draw(G, pos, ax=ax, with_labels=True, labels=labels,
            node_color=colors, node_size=2000, font_size=8,
            font_weight='bold', arrows=True, arrowsize=20,
            edge_color='#546E7A', width=1.5)

    legend_elements = [plt.Line2D([0], [0], marker='o', color='w',
                                   markerfacecolor=c, markersize=12, label=t.title())
                       for t, c in color_map.items()]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=10)
    ax.set_title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_model_graph(mlp_model, "Two-Layer MLP — ONNX Graph")

In [ ]:
def print_proto_tree(model: onnx.ModelProto, indent: int = 0) -> None:
    prefix = "  " * indent
    print(f"{prefix}ModelProto")
    print(f"{prefix}+-- ir_version: {model.ir_version}")
    print(f"{prefix}+-- producer: {model.producer_name} v{model.producer_version}")
    for oi in model.opset_import:
        d = oi.domain or '(default)'
        print(f"{prefix}+-- opset: {d} v{oi.version}")

    g = model.graph
    print(f"{prefix}+-- GraphProto: {g.name!r}")
    print(f"{prefix}    +-- inputs ({len(g.input)}):")
    for inp in g.input:
        shape = []
        if inp.type.tensor_type.HasField('shape'):
            for d in inp.type.tensor_type.shape.dim:
                shape.append(d.dim_param if d.dim_param else d.dim_value)
        print(f"{prefix}    |   +-- {inp.name}: {shape}")

    print(f"{prefix}    +-- outputs ({len(g.output)}):")
    for out in g.output:
        print(f"{prefix}    |   +-- {out.name}")

    print(f"{prefix}    +-- initializers ({len(g.initializer)}):")
    for init in g.initializer:
        dims = list(init.dims)
        nbytes = len(init.raw_data) if init.raw_data else 0
        print(f"{prefix}    |   +-- {init.name}: {dims} ({nbytes} bytes)")

    print(f"{prefix}    +-- nodes ({len(g.node)}):")
    for i, n in enumerate(g.node):
        connector = '+' if i == len(g.node) - 1 else '+'
        attrs = [f"{a.name}" for a in n.attribute]
        attr_str = f" [{', '.join(attrs)}]" if attrs else ""
        print(f"{prefix}        {connector}-- {n.op_type}: {list(n.input)} -> {list(n.output)}{attr_str}")

print_proto_tree(mlp_model)

## 10. Building a Complete Model Programmatically

Let's build a complete CNN model from scratch using the `onnx.helper` API, demonstrating every level of the proto hierarchy.

### Architecture

A small CNN for MNIST-like data:

$$\text{Input}[N, 1, 28, 28] \xrightarrow{\text{Conv}(3\times3, 16)} [N, 16, 28, 28] \xrightarrow{\text{ReLU}} \xrightarrow{\text{MaxPool}(2)} [N, 16, 14, 14]$$

$$\xrightarrow{\text{Flatten}} [N, 3136] \xrightarrow{\text{MatMul} + \text{Add}} [N, 10]$$

### Construction Steps

```
Step 1: Define ValueInfoProto (inputs/outputs)
  │
Step 2: Create TensorProto (initializers/weights)
  │
Step 3: Build NodeProto list (topological order)
  │
Step 4: Assemble GraphProto
  │
Step 5: Wrap in ModelProto with opset imports
  │
Step 6: Validate with checker.check_model()
```

In [ ]:
X_in = helper.make_tensor_value_info("input", TensorProto.FLOAT, ["batch", 1, 28, 28])
Y_out = helper.make_tensor_value_info("output", TensorProto.FLOAT, ["batch", 10])

conv_W = numpy_helper.from_array(
    np.random.randn(16, 1, 3, 3).astype(np.float32) * 0.1, name="conv.weight")
conv_B = numpy_helper.from_array(
    np.zeros(16, dtype=np.float32), name="conv.bias")
fc_W = numpy_helper.from_array(
    np.random.randn(3136, 10).astype(np.float32) * 0.01, name="fc.weight")
fc_B = numpy_helper.from_array(
    np.zeros(10, dtype=np.float32), name="fc.bias")
shape_const = numpy_helper.from_array(
    np.array([0, -1], dtype=np.int64), name="flatten_shape")

nodes = [
    helper.make_node("Conv", ["input", "conv.weight", "conv.bias"], ["conv_out"],
                     kernel_shape=[3, 3], pads=[1, 1, 1, 1], strides=[1, 1], name="conv1"),
    helper.make_node("Relu", ["conv_out"], ["relu_out"], name="relu1"),
    helper.make_node("MaxPool", ["relu_out"], ["pool_out"],
                     kernel_shape=[2, 2], strides=[2, 2], name="pool1"),
    helper.make_node("Reshape", ["pool_out", "flatten_shape"], ["flat"], name="flatten"),
    helper.make_node("MatMul", ["flat", "fc.weight"], ["mm_out"], name="fc_mm"),
    helper.make_node("Add", ["mm_out", "fc.bias"], ["output"], name="fc_add"),
]

graph = helper.make_graph(
    nodes, "mnist_cnn",
    inputs=[X_in], outputs=[Y_out],
    initializer=[conv_W, conv_B, fc_W, fc_B, shape_const],
)

cnn_model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
cnn_model.ir_version = 9
cnn_model.producer_name = "tutorial_builder"
cnn_model.producer_version = "1.0"
cnn_model.doc_string = "Minimal CNN for MNIST classification"

checker.check_model(cnn_model)
print("Model built and validated!")
print(f"  Nodes: {len(cnn_model.graph.node)}")
print(f"  Initializers: {len(cnn_model.graph.initializer)}")
total_params = sum(np.prod(list(init.dims)) for init in cnn_model.graph.initializer
                   if init.data_type == TensorProto.FLOAT)
print(f"  Total float params: {total_params:,}")
print(f"  Serialized size: {len(cnn_model.SerializeToString()):,} bytes")

print_proto_tree(cnn_model)

In [ ]:
try:
    from onnx.reference import ReferenceEvaluator
    ev = ReferenceEvaluator(cnn_model)
    x = np.random.randn(2, 1, 28, 28).astype(np.float32)
    result = ev.run(None, {"input": x})[0]
    print(f"Input shape:  {x.shape}")
    print(f"Output shape: {result.shape}")
    print(f"Output[0]:    {result[0][:5]}... (first 5 logits)")
except Exception as e:
    print(f"ReferenceEvaluator: {e}")

## 11. Formal Specification and Invariants

### Notation

Let $M$ denote a `ModelProto`, $G = M.\text{graph}$ its graph, and $\{n_1, \ldots, n_k\}$ the nodes. The ONNX spec imposes these invariants:

### Invariant 1 — Topological Order

Nodes are listed such that for every tensor name $t$ consumed by node $n_j$, either:
- $t \in \text{inputs}(G)$ (it is a graph input), or
- $\exists\, n_i$ with $i < j$ such that $t \in \text{outputs}(n_i)$ (produced by a prior node)

$$\forall j,\; \forall t \in \text{inputs}(n_j): \quad t \in \text{inputs}(G) \;\lor\; \exists\, i < j: t \in \text{outputs}(n_i)$$

### Invariant 2 — Unique Production

Within a graph, every tensor name is produced by **at most one** source:

$$\forall t: \quad |\{\text{source} \mid t \in \text{outputs}(\text{source})\}| \leq 1$$

### Invariant 3 — Type Consistency

For each tensor name $t$, the producer's output type must be consistent with every consumer's expected input type according to the operator schema:

$$\forall t,\; \forall (n_p, n_c) \text{ where } n_p \text{ produces } t \text{ and } n_c \text{ consumes } t:$$
$$\text{out\_type}(n_p, t) \sim \text{in\_type}(n_c, t)$$

### Invariant 4 — OpSet Coverage

For each node $n_i$, the `(domain, op_type)` pair must exist in an opset version $\leq$ the version declared in `ModelProto.opset_import` for that domain:

$$\forall n_i: \quad \text{since\_version}(n_i.\text{op\_type},\; n_i.\text{domain}) \leq v_{n_i.\text{domain}}$$

These invariants are checked by `onnx.checker.check_model()`, which validates the model against the formal spec.

In [ ]:
def verify_topological_order(model: onnx.ModelProto) -> bool:
    """Manually verify topological ordering of graph nodes."""
    g = model.graph
    available = set()
    for inp in g.input:
        available.add(inp.name)
    for init in g.initializer:
        available.add(init.name)

    for i, node in enumerate(g.node):
        for inp_name in node.input:
            if inp_name and inp_name not in available:
                print(f"  VIOLATION at node {i} ({node.op_type}): "
                      f"input '{inp_name}' not yet produced")
                return False
        for out_name in node.output:
            available.add(out_name)

    print("  Topological order: VALID")
    return True

def verify_unique_names(model: onnx.ModelProto) -> bool:
    g = model.graph
    producers = {}
    for inp in g.input:
        producers[inp.name] = 'graph_input'
    for i, node in enumerate(g.node):
        for out in node.output:
            if out in producers:
                print(f"  DUPLICATE: '{out}' produced by both {producers[out]} and node {i}")
                return False
            producers[out] = f'node_{i}_{node.op_type}'
    print("  Unique names: VALID")
    return True

print("=== Manual Invariant Verification ===")
verify_topological_order(cnn_model)
verify_unique_names(cnn_model)
print("\n=== Official checker ===")
try:
    checker.check_model(cnn_model)
    print("  check_model: PASSED")
except Exception as e:
    print(f"  check_model: FAILED - {e}")

## 12. Key Takeaways

1. **ONNX models are nested Protobuf messages**: `ModelProto` $\supset$ `GraphProto` $\supset$ `NodeProto` / `TensorProto` / `ValueInfoProto`, forming a well-defined tree that maps directly to the Python API.

2. **ModelProto** provides versioning (`ir_version`, `opset_import`), provenance (`producer_name`), and the computation graph. The opset import determines which operator schemas apply.

3. **GraphProto** defines a DAG where nodes are operations and edges are named tensors. Nodes must be in topological order. Required feeds are $\text{inputs} \setminus \text{initializer\_names}$.

4. **NodeProto** specifies `(domain, op_type)` with positional inputs/outputs and typed `AttributeProto` entries for static configuration. Behavior depends on the imported opset version.

5. **TensorProto** stores weight data in `raw_data` (compact binary) with size $s \cdot \prod d_i$ bytes, and supports external data references for models > 2GB.

6. **ValueInfoProto + TypeProto** provide tensor shapes with both fixed dimensions and symbolic parameters for dynamic axes.

7. **Message size** follows $S = \sum_i (|\text{tag}_i| + |\text{len}_i| + |\text{data}_i|)$, with weight data dominating for large models.

8. The formal specification enforces **topological ordering**, **unique tensor names**, **type consistency**, and **opset coverage** — all checkable via `onnx.checker.check_model()`.